In [1]:
import os, re, json, math, time, glob, hashlib
from collections import Counter
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata, files

API_KEY = userdata.get("OPENROUTER_API_KEY")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

# ---- COMMENT OUT every model except the one you are evaluating ----
MODELS = {
    # "gpt-4o-mini":           "openai/gpt-4o-mini",
    # "gemini-3.1-flash-lite": "google/gemini-3.1-flash-lite",
    "grok-4.3":              "x-ai/grok-4.3",
    # "qwen2.5-72b":           "qwen/qwen-2.5-72b-instruct",
    # "llama-3.1-70b":         "meta-llama/llama-3.1-70b-instruct",
    # "deepseek-v4-flash":     "deepseek/deepseek-v4-flash",
}

# ---- FROZEN: identical to the baseline run. Do not change. ----
INPUT_PATH   = "hallucinated_full.csv"
SEED         = 42
N_RUNS       = 1
MAX_ATTEMPTS = 20
MAX_WORKERS  = 8
MAX_TOKENS   = 8000
TEMPERATURE  = 0

N_ITEMS = None   # dry run: set 6, check the output, then set back to None

In [ ]:
if not os.path.exists(INPUT_PATH):
    files.upload()          # the exact same hallucinated_full.csv for everyone

df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

def split_steps(t):
    return [l.strip() for l in str(t).split("\n") if l.strip()]

df["_idx"]   = df["Source Index"].astype(int)
df["_steps"] = df["Hallucinated Explanation"].apply(split_steps)

lab = df.set_index("_idx")[["Hallucination Type", "Hallucination Step"]].copy()
lab["Hallucination Step"] = lab["Hallucination Step"].astype(int)

sample = df.set_index("_idx").sample(frac=1, random_state=SEED)   # shuffled; staged runs use nested subsets
if N_ITEMS: sample = sample.head(N_ITEMS)

DATA_ID = hashlib.md5(open(INPUT_PATH, "rb").read()).hexdigest()[:8]
print(len(sample), "items | data id:", DATA_ID)    # expect 608

In [ ]:
PROMPT_TEMPLATE = """You are checking a medical-calculator solution for hallucinations. A hallucination is a step that states something unsupported by, or contradicting, the patient note, the question, or correct medical knowledge.
Hallucination types:
A (knowledge hallucination): the step states a wrong equation or scoring rule for this calculator.
B (extraction hallucination): the step states a wrong value or attribute for the patient, one that conflicts with the patient note.
C (computation hallucination): the step uses the correct formula and inputs but states a wrong arithmetic result.
If the solution contains no hallucination, set "hallucinated" to false and the other fields to null.

Patient note: {note}
Question: {question}
Explanation (numbered steps):
{explanation}

Return JSON only:
{{"hallucinated": true/false, "first_hallucinated_step": <int or null>, "type": "A"/"B"/"C"/null, "reason": "<one sentence>"}}"""

CONFIG_ID = hashlib.md5((PROMPT_TEMPLATE + str(MAX_TOKENS) + str(TEMPERATURE) + str(N_RUNS)
                         + str(MAX_ATTEMPTS) + DATA_ID).encode()).hexdigest()[:8]
print("config id:", CONFIG_ID, "(must be identical for everyone)")

def call_model(slug, prompt, max_retries=3):
    """One attempt. Retries transient HTTP errors internally."""
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
               "X-Title": "MedCalc-BN hallucination eval"}
    payload = {"model": slug, "messages": [{"role": "user", "content": prompt}],
               "max_tokens": MAX_TOKENS, "temperature": TEMPERATURE,
               "response_format": {"type": "json_object"}}
    last = ""
    for i in range(max_retries):
        try:
            r = requests.post(API_URL, headers=headers, json=payload, timeout=300)
        except requests.RequestException as e:
            last = str(e); time.sleep(2 ** i * 2); continue
        if r.status_code == 400 and ("response_format" in payload or "temperature" in payload):
            payload.pop("response_format", None); payload.pop("temperature", None)
            last = r.text[:200]; continue
        if r.status_code in (429, 500, 502, 503, 504):
            last = str(r.status_code); time.sleep(2 ** i * 3); continue
        if not r.ok:
            return {"error": f"{r.status_code}: {r.text[:300]}"}
        data = r.json()
        if "choices" not in data:
            last = str(data.get("error", ""))[:200]; time.sleep(2 ** i * 3); continue
        ch = data["choices"][0]
        return {"content": ch["message"].get("content") or "",
                "finish_reason": ch.get("finish_reason"),
                "served_model": data.get("model"), "usage": data.get("usage", {})}
    return {"error": f"failed after retries: {last}"}

def to_bool(x):
    if isinstance(x, bool): return x
    if isinstance(x, str):
        s = x.strip().lower()
        if s in ("true", "yes"):  return True
        if s in ("false", "no"):  return False
    return None

def parse_verdict(text):
    try:
        t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
        obj = json.loads(re.search(r"\{.*\}", t, re.S).group(0))
        h = to_bool(obj.get("hallucinated"))
        if h is None: return None
        step = obj.get("first_hallucinated_step")
        step = int(step) if isinstance(step, (int, float)) or (isinstance(step, str) and step.strip().isdigit()) else None
        typ = obj.get("type")
        typ = typ.strip().upper() if isinstance(typ, str) and typ.strip().upper() in {"A", "B", "C"} else None
        return {"hallucinated": h, "first_hallucinated_step": step, "type": typ,
                "reason": str(obj.get("reason", ""))[:500]}
    except Exception:
        return None

In [ ]:
def raw_path(name): return f"halluc_raw_{name}.jsonl"

def one_attempt(name, idx, run, attempt):
    row = sample.loc[idx]
    prompt = PROMPT_TEMPLATE.format(note=row["Patient Note"], question=row["Question"],
                                    explanation="\n".join(row["_steps"]))
    res = call_model(MODELS[name], prompt)
    rec = {"model": name, "slug": MODELS[name], "index": int(idx), "run": run, "attempt": attempt,
           "config_id": CONFIG_ID, "data_id": DATA_ID}
    if "error" in res:
        rec.update(status="api_error", error=res["error"])
    else:
        v = parse_verdict(res["content"])
        rec.update(status="ok" if v else "parse_fail", raw=res["content"][:3000],
                   finish_reason=res["finish_reason"], served_model=res["served_model"],
                   usage=res["usage"], **(v or {}))
    return rec

def work(task):
    name, idx, run, prior = task
    recs = []
    for attempt in range(prior + 1, MAX_ATTEMPTS + 1):
        rec = one_attempt(name, idx, run, attempt)
        recs.append(rec)
        if rec["status"] == "ok": break
        time.sleep(min(2 * attempt, 10))
    return recs

ok_done, attempts = set(), Counter()
for name in MODELS:
    if os.path.exists(raw_path(name)):
        for line in open(raw_path(name), encoding="utf-8"):
            d = json.loads(line); k = (d["model"], d["index"], d["run"])
            attempts[k] += 1
            if d["status"] == "ok": ok_done.add(k)

tasks = []
for n in MODELS:
    for i in sample.index:
        for r in range(N_RUNS):
            k = (n, int(i), r)
            if k not in ok_done and attempts[k] < MAX_ATTEMPTS:
                tasks.append((n, int(i), r, attempts[k]))
print(f"{len(tasks)} items to run")

handles = {n: open(raw_path(n), "a", encoding="utf-8") for n in MODELS}
try:
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futs = [ex.submit(work, t) for t in tasks]
        for i, fut in enumerate(as_completed(futs), 1):
            for rec in fut.result():
                handles[rec["model"]].write(json.dumps(rec, ensure_ascii=False) + "\n")
            for h in handles.values(): h.flush()
            if i % 25 == 0 or i == len(futs): print(f"{i}/{len(futs)}")
finally:
    for h in handles.values(): h.close()

In [ ]:
KEY = ["model", "index", "run"]

def wilson(k, n, z=1.96):
    if n == 0: return float("nan"), float("nan")
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def accuracy_table(res, indices):
    indices = sorted(int(i) for i in indices)
    r = res[res["index"].isin(indices)]
    valid = r[r.status == "ok"].drop_duplicates(KEY, keep="first")
    models = list(res["model"].unique())
    items = pd.DataFrame([(m, i, k) for m in models for i in indices for k in range(N_RUNS)], columns=KEY)
    att = r.groupby(KEY).size().rename("attempts").reset_index()
    items = items.merge(att, on=KEY, how="left").fillna({"attempts": 0})
    items = items.merge(valid[KEY + ["hallucinated"]], on=KEY, how="left")
    items["state"] = "pending"
    items.loc[items["hallucinated"].notna(), "state"] = "valid"
    items.loc[items["hallucinated"].isna() & (items["attempts"] >= MAX_ATTEMPTS), "state"] = "excluded"
    vk = {m: set(zip(g.loc[g.state == "valid", "index"], g.loc[g.state == "valid", "run"]))
          for m, g in items.groupby("model")}
    common = set.intersection(*vk.values())
    rows = []
    for name, g in items.groupby("model", sort=False):
        v = g[g.state == "valid"]
        n_valid, correct = len(v), int((v["hallucinated"] == True).sum())   # one point per detected hallucination
        lo, hi = wilson(correct, n_valid)
        vc = v[[(i, k) in common for i, k in zip(v["index"], v["run"])]]
        rows.append({
            "model": name, "items": len(g), "valid": n_valid,
            "excluded": int((g.state == "excluded").sum()),
            "pending": int((g.state == "pending").sum()),
            "correct": correct,
            "accuracy": f"{correct / n_valid:.1%}" if n_valid else None,
            "95% CI": f"{lo:.1%}-{hi:.1%}",
            "accuracy (common items)": f"{(vc['hallucinated'] == True).mean():.1%}" if len(vc) else None,
            "avg attempts": round(g.loc[g.attempts > 0, "attempts"].mean(), 2),
            "truncated": int((r[r.model == name].finish_reason == "length").sum()),
        })
    return pd.DataFrame(rows).sort_values("correct", ascending=False)

res = pd.concat([pd.DataFrame([json.loads(l) for l in open(raw_path(n), encoding="utf-8")])
                 for n in MODELS if os.path.exists(raw_path(n))], ignore_index=True)
print(accuracy_table(res, set(sample.index)).to_string(index=False))

u = res["usage"].apply(lambda x: x if isinstance(x, dict) else {})
print("\ntokens:", res.assign(p=u.apply(lambda x: x.get("prompt_tokens", 0)),
                              c=u.apply(lambda x: x.get("completion_tokens", 0)))
                      .groupby("model")[["p", "c"]].sum().to_dict("index"))
print("served model IDs:", res.groupby("model")["served_model"].agg(lambda s: sorted(set(s.dropna()))).to_dict())

In [ ]:
paths = sorted(glob.glob("halluc_raw_*.jsonl"))
print("raw files found:", paths)
res_all = pd.concat([pd.DataFrame([json.loads(l) for l in open(p, encoding="utf-8")]) for p in paths], ignore_index=True)

ids = res_all.groupby("model")["config_id"].agg(lambda s: sorted(set(s.dropna()))).to_dict()
print("config ids per model:", ids)
if len({x for v in ids.values() for x in v}) != 1:
    print("WARNING: runs used different settings or data. Do not compare these models.")

final = accuracy_table(res_all, set(df["_idx"]))
print(final.to_string(index=False))
final.to_csv("halluc_eval_summary.csv", index=False)
files.download("halluc_eval_summary.csv")

In [ ]:
for n in MODELS:
    files.download(raw_path(n))